# 03. Finetune Spot-check (Image × Regression, ADR-019 보조)

probe proxy 신뢰성 검증용. ResNet18을 **blur × 5레벨 + baseline**에서만 finetune(epoch 5) → clean test R².
재오염(시드 고정) 후 학습. num_workers=0 + 반복마다 정리. 결과는 method='finetune'으로 append.

In [ ]:
# 0-1. Drive 마운트 + GPU
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, gc
import numpy as np, pandas as pd, torch
BASE = '/content/drive/MyDrive/capstone/dsc'
RESULTS_DIR = f'{BASE}/results'
DATA_DIR = f'{BASE}/data/image_regression'
os.makedirs(RESULTS_DIR, exist_ok=True); os.makedirs(DATA_DIR, exist_ok=True)
if BASE not in sys.path: sys.path.insert(0, BASE)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {device} | torch {torch.__version__}')

In [ ]:
%pip install -q datasets timm imagehash opencv-python-headless

In [ ]:
# 사전등록 메타 (ADR-018/019)
DATASETS = {
    'UTKFace':      {'hf': 'Subh775/UTKFace_demographics_V1', 'target': 'age',          'image_col': 'image', 'role': 'tune'},
    'SCUT_FBP5500': {'hf': 'MnLgt/scut-fbp5500',              'target': 'beauty_score', 'image_col': 'image', 'role': 'held-out'},
}
TUNE_DS, HELD_DS = 'UTKFace', 'SCUT_FBP5500'
POLLUTION_LEVELS = [0.1, 0.3, 0.5, 0.7, 0.9]
SAMPLE_CAP = 2000
TEST_CAP = 2000
RANDOM_SEED = 42; ML_SPLIT_SEED = 1; ML_TEST_SIZE = 0.2
print('datasets:', list(DATASETS.keys()), '| levels:', POLLUTION_LEVELS)
SPOT_POLLUTER = 'blur'
EPOCHS = 5; BATCH_SIZE = 128; LR = 1e-3; IMAGE_SIZE = 224
print('spot-check:', SPOT_POLLUTER, 'levels', POLLUTION_LEVELS, 'epochs', EPOCHS)

In [ ]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split
def load_hf_split(ds_name):
    meta = DATASETS[ds_name]
    ds = load_dataset(meta['hf'], split='train')
    tr_idx, te_idx = train_test_split(np.arange(len(ds)), test_size=ML_TEST_SIZE, random_state=ML_SPLIT_SEED)
    return ds, meta, tr_idx, te_idx
def to_arrays(ds, meta, indices, sample_cap=None, random_state=1):
    indices = np.asarray(indices)
    if sample_cap and len(indices) > sample_cap:
        rng = np.random.RandomState(random_state)
        indices = indices[rng.permutation(len(indices))[:sample_cap]]
    images, targets = [], []
    for i in indices:
        ex = ds[int(i)]; img = ex[meta['image_col']]
        if hasattr(img, 'convert'): img = img.convert('RGB')
        images.append(np.array(img, dtype=np.uint8)); targets.append(float(ex[meta['target']]))
    return images, targets

In [ ]:
# 모델/학습/재오염 import
import importlib
if BASE not in sys.path: sys.path.insert(0, BASE)
importlib.invalidate_caches()
for _m in list(sys.modules):
    if _m.startswith('dsc_framework'): del sys.modules[_m]
if not hasattr(pd.DataFrame, 'append'):
    pd.DataFrame.append = lambda s, o, ignore_index=False, **k: pd.concat([s, o], ignore_index=ignore_index)
from dsc_framework.image_polluters import BlurPolluter
import torch.nn as nn
import torchvision.models as tvm
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import r2_score
def get_resnet18_reg():
    m = tvm.resnet18(weights=tvm.ResNet18_Weights.IMAGENET1K_V1)
    m.fc = nn.Linear(m.fc.in_features, 1); return m
class RegDS(Dataset):
    def __init__(self, images, targets, tf): self.images, self.targets, self.tf = images, targets, tf
    def __len__(self): return len(self.images)
    def __getitem__(self, i):
        from PIL import Image
        a = np.asarray(self.images[i], dtype=np.uint8)
        a = a.squeeze() if a.ndim == 3 and a.shape[-1] == 1 else a
        img = Image.fromarray(a) if a.ndim == 2 else Image.fromarray(a[..., :3])
        if img.mode != 'RGB': img = img.convert('RGB')
        return self.tf(img), float(self.targets[i])
tf = T.Compose([T.Resize((IMAGE_SIZE, IMAGE_SIZE)), T.ToTensor(),
                T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])])
def finetune_eval(tr_img, tr_tgt, te_img, te_tgt, epochs=EPOCHS):
    model = get_resnet18_reg().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR); crit = nn.MSELoss()
    tl = DataLoader(RegDS(tr_img, tr_tgt, tf), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    vl = DataLoader(RegDS(te_img, te_tgt, tf), batch_size=BATCH_SIZE, num_workers=0)
    for _ in range(epochs):
        model.train()
        for x, y in tl:
            x, y = x.to(device), y.to(device).float().unsqueeze(1)
            opt.zero_grad(); crit(model(x), y).backward(); opt.step()
    model.eval(); preds, trues = [], []
    with torch.no_grad():
        for x, y in vl:
            preds.append(model(x.to(device)).cpu().numpy().ravel()); trues.append(np.asarray(y))
    r2 = r2_score(np.concatenate(trues), np.concatenate(preds))
    del model; gc.collect(); torch.cuda.empty_cache()
    return max(0.0, r2)
print('spot-check 함수 정의 완료')

In [ ]:
# spot-check 루프: blur × levels + baseline, ResNet18 finetune
from time import time
perf_path = f'{RESULTS_DIR}/model_performance_image_regression.csv'
df = pd.read_csv(perf_path) if os.path.isfile(perf_path) else pd.DataFrame()
done = set()
if len(df) and 'method' in df.columns:
    ft = df[df.method == 'finetune']
    done = set(ft.apply(lambda r: f"{r['dataset']}|{r['polluter']}|{r['level']}", axis=1))
rows = df.to_dict('records'); t_all = time()
for ds_name in DATASETS:
    ds, meta, tr_idx, te_idx = load_hf_split(ds_name)
    tr_img, tr_tgt = to_arrays(ds, meta, tr_idx, sample_cap=SAMPLE_CAP, random_state=1)
    te_img, te_tgt = to_arrays(ds, meta, te_idx, sample_cap=TEST_CAP, random_state=1)
    del ds; gc.collect()
    configs = [('none', 0.0)] + [(SPOT_POLLUTER, lv) for lv in POLLUTION_LEVELS]
    for pname, level in configs:
        key = f'{ds_name}|{pname}|{level}'
        if key in done: print('skip', key); continue
        t0 = time()
        if pname == 'none':
            pi, pt = tr_img, tr_tgt
        else:
            pi, pt = BlurPolluter(level=level, random_seed=RANDOM_SEED).pollute(tr_img, tr_tgt)
        try:
            r2 = finetune_eval(pi, pt, te_img, te_tgt)
            rows.append({'dataset': ds_name, 'polluter': pname, 'level': level,
                         'method': 'finetune', 'model': 'ResNet18', 'score': round(r2, 4)})
            print(f'  {ds_name}/{pname}_{int(level*100)} finetune R2={r2:+.4f} ({time()-t0:.0f}s)')
        except Exception as e:
            print(f'  {ds_name}/{pname}_{int(level*100)} ERROR: {e}')
        if pname != 'none': del pi, pt
        gc.collect()
        pd.DataFrame(rows).to_csv(perf_path, index=False)
    del tr_img, tr_tgt, te_img, te_tgt; gc.collect()
print(f'\nspot-check 완료 ({time()-t_all:.0f}s) → 04')